<a href="https://colab.research.google.com/github/zydanne-costa/Ondas_ADCP_SCO_Mar_Nov_2025/blob/main/DSPEC_Refinamento.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ============================================================
# IMPORTAÇÃO DAS BIBLIOTECAS
# ============================================================

import os
import glob
import numpy as np
import pandas as pd

from datetime import datetime

from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# ============================================================
# DIRETÓRIOS
# ============================================================

chu_dir = "/content/drive/MyDrive/Ondas/Dados/marco2025/SCO1/DSpec_2/"

sec_dir = "/content/drive/MyDrive/Ondas/Dados/novembro2025/SCO2/DSpec_2/"

output_dir = "/content/drive/MyDrive/Ondas/Dados/Refinados/DSpec/"

os.makedirs(output_dir, exist_ok=True)

In [ ]:
# ============================================================
# CONFIGURAÇÃO DO ESPECTRO
# ============================================================

N_FREQ = 64
N_DIR = 90

FREQ_INICIAL = 0.00878906
DELTA_F = 0.015625

frequencias = FREQ_INICIAL + np.arange(N_FREQ) * DELTA_F

direcoes = np.linspace(
    0,
    360,
    N_DIR,
    endpoint=False
)

In [ ]:
# ============================================================
# EXTRAÇÃO DOS PARÂMETROS ESPECTRAIS
# ============================================================

def processar_dspec(diretorio):

    arquivos = sorted(
        glob.glob(
            os.path.join(diretorio, "*.txt")
        )
    )

    resultados = []

    print(f"{len(arquivos)} arquivos encontrados.\n")

    for arquivo in arquivos:

        nome = os.path.basename(arquivo)

        # ----------------------------------------------------
        # Leitura da matriz espectral
        # ----------------------------------------------------

        energia = np.loadtxt(
            arquivo,
            comments="%"
        )

        energia = energia / 1e6

        # ----------------------------------------------------
        # Data e hora
        # ----------------------------------------------------

        codigo = nome.replace("DSpec","").replace(".txt","")

        datahora = datetime.strptime(
            codigo,
            "%y%m%d%H%M"
        )

        # ----------------------------------------------------
        # Frequência de pico
        # ----------------------------------------------------

        # Limite da faixa de frequências utilizada
        fmin = 0.05
        fmax = 0.50

        # Energia integrada em cada frequência
        energia_freq = energia.sum(axis=1)

        # Seleciona apenas a faixa de interesse
        mascara = (frequencias >= fmin) & (frequencias <= fmax)

        freq_validas = frequencias[mascara]
        energia_valida = energia_freq[mascara]

        # Índice do pico dentro da faixa válida
        indice_local = np.argmax(energia_valida)

        # Índice correspondente na matriz original
        indice_fp = np.where(mascara)[0][indice_local]

        # Frequência e período de pico
        fp = frequencias[indice_fp]
        Tp = 1 / fp

        # ----------------------------------------------------
        # Direção predominante na frequência de pico
        # ----------------------------------------------------

        energia_direcional = energia[indice_fp, :]

        indice_dp = np.argmax(energia_direcional)

        Dp = direcoes[indice_dp]

        # ----------------------------------------------------
        # Armazenar resultados
        # ----------------------------------------------------

        resultados.append({

            "Arquivo": nome,

            "DataHora": datahora,

            "fp_Hz": fp,

            "Tp_s": Tp,

            "Dp_deg": Dp

        })

    df = pd.DataFrame(resultados)

    df["fp_Hz"] = df["fp_Hz"].round(5)

    df["Tp_s"] = df["Tp_s"].round(2)

    df["Dp_deg"] = df["Dp_deg"].round(1)

    return df

In [ ]:
# ============================================================
# PROCESSAMENTO
# ============================================================

df_chuvoso = processar_dspec(chu_dir)

df_seco = processar_dspec(sec_dir)

442 arquivos encontrados.

168 arquivos encontrados.



In [ ]:
# ============================================================
# CÁLCULO DO Dp A CADA 1 HORA (MODA)
# ============================================================

def calcular_dp_1h(df):

    df = df.copy()

    df["DataHora"] = pd.to_datetime(df["DataHora"])

    df = df.set_index("DataHora")

    # Moda da direção predominante dentro de cada hora
    dp_1h = (
        df["Dp_deg"]
        .resample("1H")
        .agg(lambda x: x.mode().iloc[0] if not x.mode().empty else np.nan)
        .reset_index()
    )

    dp_1h.columns = ["DataHora", "Dp_deg"]

    return dp_1h

dp_1h_chu = calcular_dp_1h(df_chuvoso)

dp_1h_sec = calcular_dp_1h(df_seco)

/tmp/ipykernel_1750/834753712.py:16: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  .resample("1H")
/tmp/ipykernel_1750/834753712.py:16: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  .resample("1H")


In [ ]:
# ============================================================
# DIAGNÓSTICO
# ============================================================

print("="*60)
print("CHUVOSO")
print("="*60)

print(df_chuvoso.head())

print()

print(df_chuvoso.describe())

print()

print("="*60)
print("MENOS CHUVOSO")
print("="*60)

print(df_seco.head())

print()

print(df_seco.describe())

CHUVOSO
               Arquivo            DataHora    fp_Hz   Tp_s  Dp_deg
0  DSpec2503271300.txt 2025-03-27 13:00:00  0.05566  17.96     0.0
1  DSpec2503271320.txt 2025-03-27 13:20:00  0.19629   5.09   280.0
2  DSpec2503271340.txt 2025-03-27 13:40:00  0.33691   2.97   128.0
3  DSpec2503271400.txt 2025-03-27 14:00:00  0.19629   5.09   120.0
4  DSpec2503271420.txt 2025-03-27 14:20:00  0.19629   5.09   128.0

                  DataHora       fp_Hz        Tp_s      Dp_deg
count                  442  442.000000  442.000000  442.000000
mean   2025-03-30 14:30:00    0.148670    8.575588  106.895928
min    2025-03-27 13:00:00    0.055660    2.160000    0.000000
25%    2025-03-29 01:45:00    0.118160    5.540000   16.000000
50%    2025-03-30 14:30:00    0.133790    7.470000   72.000000
75%    2025-04-01 03:15:00    0.180660    8.460000  163.000000
max    2025-04-02 16:00:00    0.461910   17.960000  356.000000
std                    NaN    0.074739    4.561942  111.151683

MENOS CHUVOSO
       

In [ ]:
# ============================================================
# EXPORTAÇÃO
# ============================================================

arquivo_chu = os.path.join(
    output_dir,
    "DSpec_Chuvoso_Refinado.txt"
)

arquivo_sec = os.path.join(
    output_dir,
    "DSpec_MenosChuvoso_Refinado.txt"
)

df_chuvoso.to_csv(
    arquivo_chu,
    sep="\t",
    index=False
)

df_seco.to_csv(
    arquivo_sec,
    sep="\t",
    index=False
)

print()

print("Arquivos exportados com sucesso.")

print(arquivo_chu)

print(arquivo_sec)


Arquivos exportados com sucesso.
/content/drive/MyDrive/Ondas/Dados/Refinados/DSpec/DSpec_Chuvoso_Refinado.txt
/content/drive/MyDrive/Ondas/Dados/Refinados/DSpec/DSpec_MenosChuvoso_Refinado.txt


In [ ]:
# ============================================================
# EXPORTAÇÃO DO Dp 1H
# ============================================================

arquivo_dp_chu = os.path.join(
    output_dir,
    "Dp_1H_CHU.txt"
)

arquivo_dp_sec = os.path.join(
    output_dir,
    "Dp_1H_SEC.txt"
)

dp_1h_chu.to_csv(
    arquivo_dp_chu,
    sep="\t",
    index=False,
    date_format="%Y-%m-%d %H:%M:%S"
)

dp_1h_sec.to_csv(
    arquivo_dp_sec,
    sep="\t",
    index=False,
    date_format="%Y-%m-%d %H:%M:%S"
)

print("Arquivos Dp 1H exportados com sucesso.")
print(arquivo_dp_chu)
print(arquivo_dp_sec)

Arquivos Dp 1H exportados com sucesso.
/content/drive/MyDrive/Ondas/Dados/Refinados/DSpec/Dp_1H_CHU.txt
/content/drive/MyDrive/Ondas/Dados/Refinados/DSpec/Dp_1H_SEC.txt


In [10]:
# ============================================================
# ADICIONAR Dp AO WAVEDATA
# ============================================================

arquivo_wave_chu = (
    "/content/drive/MyDrive/Ondas/Dados/Refinados/WaveData/wwdata_chu.txt"
)

arquivo_wave_sec = (
    "/content/drive/MyDrive/Ondas/Dados/Refinados/WaveData/wwdata_sec.txt"
)

wave_chu = pd.read_csv(
    arquivo_wave_chu,
    sep="\t"
)

wave_sec = pd.read_csv(
    arquivo_wave_sec,
    sep="\t"
)

wave_chu["DataHora"] = pd.to_datetime(wave_chu["DataHora"])
wave_sec["DataHora"] = pd.to_datetime(wave_sec["DataHora"])

dp_1h_chu["DataHora"] = pd.to_datetime(dp_1h_chu["DataHora"])
dp_1h_sec["DataHora"] = pd.to_datetime(dp_1h_sec["DataHora"])

wave_chu = wave_chu.merge(
    dp_1h_chu,
    on="DataHora",
    how="left"
)

wave_sec = wave_sec.merge(
    dp_1h_sec,
    on="DataHora",
    how="left"
)

In [12]:
# ============================================================
# SALVAR WAVEDATA COM Dp
# ============================================================

wave_chu.to_csv(
    arquivo_wave_chu,
    sep="\t",
    index=False,
    date_format="%Y-%m-%d %H:%M:%S"
)

wave_sec.to_csv(
    arquivo_wave_sec,
    sep="\t",
    index=False,
    date_format="%Y-%m-%d %H:%M:%S"
)

print("WaveData atualizado com Dp.")

WaveData atualizado com Dp.
